<a href="https://colab.research.google.com/github/PsicoJazz/Ciencia-de-Dados/blob/main/dataSUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Atualiza as ferramentas de empacotamento fundamentais
!pip install --upgrade pip setuptools wheel

# 2. Instala a biblioteca do PySUS e o Pandas para manipulação dos dados
!pip install pysus pandas


In [17]:
import pandas as pd
# Nova forma de importar: os dados de mortalidade são acessados direto pela raiz ou por subnamespaces dedicados
import pysus

# 1. Definir os códigos IBGE dos municípios da Região de Marília
MAPA_REGIAO = {
    352900: "Marília"
}
CODIGOS_REGIAO = list(MAPA_REGIAO.keys())
ANO = 2024  # Altere para o ano desejado

print(f"Iniciando download dos dados do SIM de SP para o ano {ANO}...")

# 2. Baixar a base do Sistema de Informações sobre Mortalidade (SIM) de São Paulo
# Na nova versão do PySUS, as funções simplificadas retornam um DataFrame pronto do Pandas automaticamente!
df_sp = pysus.ftp.sim(state="SP", year=ANO).to_dataframe()

print("Base de dados carregada do DATASUS! Filtrando dados da região de Marília...")

# 3. Filtrar pelas cidades da região de Marília
df_sp['CODMUNRES'] = pd.to_numeric(df_sp['CODMUNRES'], errors='coerce')
df_regiao = df_sp[df_sp['CODMUNRES'].isin(CODIGOS_REGIAO)].copy()

# Mapeia o nome do município para a nova coluna
df_regiao['NOME_MUNICIPIO'] = df_regiao['CODMUNRES'].map(MAPA_REGIAO)

# 4. Filtrar pelas Doenças Crônicas Não Transmissíveis (CID-10)
# E10-E14: Diabetes | I00-I99: Aparelho Circulatório | J40-J47: Respiratórias Crônicas
def filtrar_cronicas(cid):
    if pd.isna(cid):
        return False
    cid = str(cid).upper()
    return cid.startswith('E1') or cid.startswith('I') or cid.startswith('J4')

# Aplica o filtro usando a causa básica do óbito (CAUSABAS)
df_regiao_cronicas = df_regiao[df_regiao['CAUSABAS'].apply(filtrar_cronicas)]

# 5. Salvar o arquivo finalizado no ambiente do Colab
nome_arquivo = f"doencas_cronicas_REGIAO_marilia_{ANO}.csv"
df_regiao_cronicas.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

print(f"\n🎉 Mineração concluída!")
print(f"Foram identificados {len(df_regiao_cronicas)} registros de doenças crônicas na Região de Marília em {ANO}.")
print(f"Arquivo disponível na aba de arquivos ao lado: {nome_arquivo}")

Iniciando download dos dados do SIM de SP para o ano 2024...


DOSP2024.parquet: 0.00B [00:00, ?B/s]


Base de dados carregada do DATASUS! Filtrando dados da região de Marília...

🎉 Mineração concluída!
Foram identificados 724 registros de doenças crônicas na Região de Marília em 2024.
Arquivo disponível na aba de arquivos ao lado: doencas_cronicas_REGIAO_marilia_2024.csv


In [19]:
import pandas as pd
doencasCronicas = pd.read_csv('dcMarilia.csv')
doencasCronicas.head()

,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,RACACOR,...,TPRESGINFO,TPNIVELINV,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,CONTADOR,NOME_MUNICIPIO
0,1,2,2012024,2055.0,835,352900.0,21021942,481,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5181,Marília
1,1,2,2012024,1100.0,831,313920.0,3051955,468,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5182,Marília
2,1,2,2012024,1624.0,825,250510.0,4091942,481,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5183,Marília
3,1,2,3012024,1750.0,831,315590.0,28061935,488,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8907,Marília
4,1,2,3012024,730.0,835,352710.0,20031932,491,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9048,Marília


In [21]:
doencasCronicas.describe()

,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,RACACOR,...,DTCONINV,FONTES,TPRESGINFO,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,CONTADOR
count,724.0,724.0,7.240000e+02,721.000000,724.000000,719.000000,7.240000e+02,724.000000,724.000000,724.000000,...,1.200000e+01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.240000e+02
mean,1.0,2.0,1.526941e+07,1236.319001,830.216851,343487.568846,1.514670e+07,474.461326,1.469613,1.796961,...,1.715286e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.569621e+05
std,0.0,0.0,8.881705e+06,679.458157,51.447740,34972.974197,8.934236e+06,13.527285,0.499421,1.229621,...,9.793583e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.265517e+05
min,1.0,2.0,1.022024e+06,0.000000,127.000000,120010.000000,1.011937e+06,419.000000,1.000000,1.000000,...,4.102024e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.181000e+03
25%,1.0,2.0,7.112024e+06,645.000000,835.000000,350700.000000,7.071971e+06,466.000000,1.000000,1.000000,...,6.847024e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.947568e+05
50%,1.0,2.0,1.507702e+07,1245.000000,835.000000,352900.000000,1.504695e+07,476.000000,1.000000,1.000000,...,1.910202e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.262930e+05
75%,1.0,2.0,2.303202e+07,1825.000000,835.000000,353460.000000,2.306194e+07,485.000000,2.000000,3.000000,...,2.726952e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.114687e+06
max,1.0,2.0,3.110202e+07,2357.000000,853.000000,530010.000000,3.110195e+07,503.000000,2.000000,5.000000,...,2.907202e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.524595e+06


In [22]:
print(doencasCronicas['CAUSABAS'].head())

0    E112
1    I110
2    I500
3    I740
4    I219
Name: CAUSABAS, dtype: object


In [23]:
print(doencasCronicas.columns.tolist())


['ORIGEM', 'TIPOBITO', 'DTOBITO', 'HORAOBITO', 'NATURAL', 'CODMUNNATU', 'DTNASC', 'IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC', 'ESC2010', 'SERIESCFAL', 'OCUP', 'CODMUNRES', 'LOCOCOR', 'CODESTAB', 'ESTABDESCR', 'CODMUNOCOR', 'IDADEMAE', 'ESCMAE', 'ESCMAE2010', 'SERIESCMAE', 'OCUPMAE', 'QTDFILVIVO', 'QTDFILMORT', 'GRAVIDEZ', 'SEMAGESTAC', 'GESTACAO', 'PARTO', 'OBITOPARTO', 'PESO', 'TPMORTEOCO', 'OBITOGRAV', 'OBITOPUERP', 'ASSISTMED', 'EXAME', 'CIRURGIA', 'NECROPSIA', 'LINHAA', 'LINHAB', 'LINHAC', 'LINHAD', 'LINHAII', 'CAUSABAS', 'CB_PRE', 'COMUNSVOIM', 'DTATESTADO', 'CIRCOBITO', 'ACIDTRAB', 'FONTE', 'NUMEROLOTE', 'TPPOS', 'DTINVESTIG', 'CAUSABAS_O', 'DTCADASTRO', 'ATESTANTE', 'STCODIFICA', 'CODIFICADO', 'VERSAOSIST', 'VERSAOSCB', 'FONTEINV', 'DTRECEBIM', 'ATESTADO', 'DTRECORIGA', 'CAUSAMAT', 'ESCMAEAGR1', 'ESCFALAGR1', 'STDOEPIDEM', 'STDONOVA', 'DIFDATA', 'NUDIASOBCO', 'NUDIASOBIN', 'DTCADINV', 'TPOBITOCOR', 'DTCONINV', 'FONTES', 'TPRESGINFO', 'TPNIVELINV', 'NUDIASINF', 'DTCADINF', 'MORT

In [24]:
# 1. Criar a lista com os nomes exatos das colunas RELEVANTES que decidimos manter
colunas_relevantes = [
    'CAUSABAS', 'CAUSABAS_O',                           # Dados da Doença (CID-10)
    'LINHAA', 'LINHAB', 'LINHAC', 'LINHAD', 'LINHAII',  # Comorbidades e linhas do atestado
    'CODMUNRES', 'NOME_MUNICIPIO', 'CODMUNOCOR',        # Localização Geográfica
    'LOCOCOR',                                          # Tipo de local (Hospital, Casa, etc.)
    'IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC2010',    # Perfil Sociodemográfico
    'OCUP',                                             # Ocupação/Trabalho
    'DTOBITO',                                          # Data do evento (Linha do tempo)
    'ASSISTMED'                                         # Teve assistência médica ou não
]

# 2. Filtrar o DataFrame original guardando o resultado em uma nova variável limpa
# O parâmetro errors='ignore' serve para o caso de alguma coluna opcional (como CAUSABAS_O) não existir no ano escolhido
doencasCronicas_limpo = doencasCronicas.filter(items=colunas_relevantes, axis=1)

# 3. Mostrar as primeiras linhas do resultado para conferir a tabela reduzida
print(f"Tabela reduzida com sucesso! Colunas mantidas: {doencasCronicas_limpo.shape[1]}")
doencasCronicas_limpo.head()


Tabela reduzida com sucesso! Colunas mantidas: 19


,CAUSABAS,CAUSABAS_O,LINHAA,LINHAB,LINHAC,LINHAD,LINHAII,CODMUNRES,NOME_MUNICIPIO,CODMUNOCOR,LOCOCOR,IDADE,SEXO,RACACOR,ESTCIV,ESC2010,OCUP,DTOBITO,ASSISTMED
0,E112,N039,*A419,*N12X,*N039,NaN,*E119*I10X*C509,352900,Marília,352900,1,481,2,3,3,1.0,999992.0,2012024,1.0
1,I110,I110,*R570,*I509,*R074,*I10X,NaN,352900,Marília,352900,1,468,1,4,4,1.0,623110.0,2012024,1.0
2,I500,I500,*J969,*J90X,*J159,*I500,NaN,352900,Marília,352900,2,481,2,1,2,1.0,999993.0,2012024,NaN
3,I740,I740,*R688,*J969,*I740,NaN,*F03X*I10X*E149,352900,Marília,352900,1,488,2,1,3,0.0,999992.0,3012024,NaN
4,I219,I219,*I219,*I10X,NaN,NaN,NaN,352900,Marília,352900,3,491,1,1,2,3.0,999993.0,3012024,1.0


In [25]:
# Dicionário de tradução das colunas do DATASUS
mapa_nomes = {
    'CAUSABAS': 'causa_principal_cid',
    'CAUSABAS_O': 'causa_principal_original_cid',
    'LINHAA': 'comorbidade_linha_a',
    'LINHAB': 'comorbidade_linha_b',
    'LINHAC': 'comorbidade_linha_c',
    'LINHAD': 'comorbidade_linha_d',
    'LINHAII': 'comorbidade_outras_causas',
    'CODMUNRES': 'codigo_municipio_residencia',
    'NOME_MUNICIPIO': 'nome_municipio',
    'CODMUNOCOR': 'codigo_municipio_ocorrencia',
    'LOCOCOR': 'local_ocorrencia_tipo',
    'IDADE': 'idade_bruta',
    'SEXO': 'genero_codigo',
    'RACACOR': 'raca_cor_codigo',
    'ESTCIV': 'estado_civil_codigo',
    'ESC2010': 'escolaridade_codigo',
    'OCUP': 'ocupacao_codigo',
    'DTOBITO': 'data_ocorrencia',
    'ASSISTMED': 'teve_assistencia_medica'
}

# Renomeia as colunas diretamente no DataFrame limpo
doencasCronicas_limpo.rename(columns=mapa_nomes, inplace=True)

# Exibe os novos nomes das colunas para conferir
print("✨ Colunas renomeadas com sucesso!")
print(doencasCronicas_limpo.columns.tolist())


✨ Colunas renomeadas com sucesso!
['causa_principal_cid', 'causa_principal_original_cid', 'comorbidade_linha_a', 'comorbidade_linha_b', 'comorbidade_linha_c', 'comorbidade_linha_d', 'comorbidade_outras_causas', 'codigo_municipio_residencia', 'nome_municipio', 'codigo_municipio_ocorrencia', 'local_ocorrencia_tipo', 'idade_bruta', 'genero_codigo', 'raca_cor_codigo', 'estado_civil_codigo', 'escolaridade_codigo', 'ocupacao_codigo', 'data_ocorrencia', 'teve_assistencia_medica']


In [27]:

import numpy as np

# 1. Remover as colunas de comorbidades e códigos brutos que não vão para o dashboard
colunas_para_remover = [
    'comorbidade_linha_a', 'comorbidade_linha_b', 'comorbidade_linha_c',
    'comorbidade_linha_d', 'comorbidade_outras_causas', 'causa_principal_original_cid',
    'codigo_municipio_residencia', 'codigo_municipio_ocorrencia'
]
# Usamos errors='ignore' caso alguma dessas já tenha sido removida antes
df_dash = doencasCronicas_limpo.drop(columns=colunas_para_remover, errors='ignore').copy()


# 2. TRATAMENTO DA IDADE (Correção do padrão DATASUS)
# No SUS: 400 a 499 significa anos (ex: 465 = 65 anos). Valores menores são meses/dias (convertemos para 0 anos).
def corrigir_idade(idade_bruta):
    try:
        id_num = int(float(idade_bruta))
        if id_num >= 400 and id_num <= 499:
            return id_num - 400
        elif id_num >= 500: # Centenários (ex: 502 = 102 anos)
            return id_num - 400
        else:
            return 0 # Menores de 1 ano (meses/dias)
    except:
        return np.nan

df_dash['idade'] = df_dash['idade_bruta'].apply(corrigir_idade)
df_dash.drop(columns=['idade_bruta'], inplace=True) # Remove a coluna confusa antiga


# 3. TRATAMENTO DO GÊNERO/SEXO
mapa_sexo = {1: 'Masculino', 2: 'Feminino', '1': 'Masculino', '2': 'Feminino'}
df_dash['genero'] = df_dash['genero_codigo'].map(mapa_sexo).fillna('Não Informado')
df_dash.drop(columns=['genero_codigo'], inplace=True)


# 4. TRATAMENTO DA RAÇA / COR
mapa_raca = {
    1: 'Branca', 2: 'Preta', 3: 'Amarela', 4: 'Parda', 5: 'Indígena',
    '1': 'Branca', '2': 'Preta', '3': 'Amarela', '4': 'Parda', '5': 'Indígena'
}
df_dash['raca_cor'] = df_dash['raca_cor_codigo'].map(mapa_raca).fillna('Não Informado')
df_dash.drop(columns=['raca_cor_codigo'], inplace=True)


# 5. TRATAMENTO DO LOCAL DE OCORRÊNCIA
mapa_local = {
    1: 'Hospital', 2: 'Outro Estab. Saúde', 3: 'Domicílio', 4: 'Via Pública', 5: 'Outros',
    '1': 'Hospital', '2': 'Outro Estab. Saúde', '3': 'Domicílio', '4': 'Via Pública', '5': 'Outros'
}
df_dash['local_ocorrencia'] = df_dash['local_ocorrencia_tipo'].map(mapa_local).fillna('Não Informado')
df_dash.drop(columns=['local_ocorrencia_tipo'], inplace=True)


# 6. TRATAMENTO DA ASSISTÊNCIA MÉDICA
mapa_assist = {1: 'Sim', 2: 'Não', '1': 'Sim', '2': 'Não'}
df_dash['teve_assistencia_medica'] = df_dash['teve_assistencia_medica'].map(mapa_assist).fillna('Não Informado')


# 7. TRATAMENTO E AGRUPAMENTO DOS CIDs (Dar nome real às doenças crônicas)
def traduzir_cid_grupo(cid):
    if pd.isna(cid):
        return 'Outras / Não Informado'
    cid = str(cid).upper().strip()

    if cid.startswith('E10') or cid.startswith('E11') or cid.startswith('E12') or cid.startswith('E13') or cid.startswith('E14'):
        return 'Diabetes Mellitus'
    elif cid.startswith('I10') or cid.startswith('I11') or cid.startswith('I12') or cid.startswith('I13') or cid.startswith('I15'):
        return 'Doenças Hipertensivas'
    elif cid.startswith('I20') or cid.startswith('I21') or cid.startswith('I22') or cid.startswith('I23') or cid.startswith('I24') or cid.startswith('I25'):
        return 'Doenças Isquêmicas do Coração (Infarto/Angina)'
    elif cid.startswith('I60') or cid.startswith('I61') or cid.startswith('I62') or cid.startswith('I63') or cid.startswith('I64') or cid.startswith('I69'):
        return 'Doenças Cerebrovasculares (AVC/Derrame)'
    elif cid.startswith('J40') or cid.startswith('J41') or cid.startswith('J42') or cid.startswith('J43') or cid.startswith('J44'):
        return 'Doenças Respiratórias Crônicas (DPOC/Bronquite/Enfisema)'
    else:
        return 'Outras Doenças Crônicas Cardiovasculares/Metabólicas'

df_dash['doenca_cronica_grupo'] = df_dash['causa_principal_cid'].apply(traduzir_cid_grupo)


# 8. AJUSTAR FORMATO DA DATA
df_dash['data_ocorrencia'] = pd.to_datetime(df_dash['data_ocorrencia'], format='%d%m%Y', errors='coerce')


# 9. REORGANIZAR AS COLUNAS FINAIS (Manter uma ordem limpa)
colunas_finais = [
    'data_ocorrencia', 'nome_municipio', 'doenca_cronica_grupo',
    'causa_principal_cid', 'idade', 'genero', 'raca_cor',
    'local_ocorrencia', 'teve_assistencia_medica'
]
df_dashboard_pronto = df_dash[colunas_finais].copy()

# Salvar o arquivo final perfeito para o Dashboard
df_dashboard_pronto.to_csv('dados_mortalidade_marilia_dashboard.csv', index=False, encoding='utf-8-sig')

print("🚀 Base de dados higienizada e PRONTA para o Dashboard!")
print(f"Colunas resultantes: {df_dashboard_pronto.columns.tolist()}")
df_dashboard_pronto.head()


🚀 Base de dados higienizada e PRONTA para o Dashboard!
Colunas resultantes: ['data_ocorrencia', 'nome_municipio', 'doenca_cronica_grupo', 'causa_principal_cid', 'idade', 'genero', 'raca_cor', 'local_ocorrencia', 'teve_assistencia_medica']


,data_ocorrencia,nome_municipio,doenca_cronica_grupo,causa_principal_cid,idade,genero,raca_cor,local_ocorrencia,teve_assistencia_medica
0,2024-01-20,Marília,Diabetes Mellitus,E112,81,Feminino,Amarela,Hospital,Sim
1,2024-01-20,Marília,Doenças Hipertensivas,I110,68,Masculino,Parda,Hospital,Sim
2,2024-01-20,Marília,Outras Doenças Crônicas Cardiovasculares/Metab...,I500,81,Feminino,Branca,Outro Estab. Saúde,Não Informado
3,2024-01-30,Marília,Outras Doenças Crônicas Cardiovasculares/Metab...,I740,88,Feminino,Branca,Hospital,Não Informado
4,2024-01-30,Marília,Doenças Isquêmicas do Coração (Infarto/Angina),I219,91,Masculino,Branca,Domicílio,Sim


In [28]:
# Exporta o DataFrame pronto para um arquivo do Excel (.xlsx)
# index=False evita que o Pandas crie uma coluna extra de numeração (0, 1, 2...) no Excel
df_dashboard_pronto.to_excel('dados_marilia_dashboard.xlsx', index=False)

print("🎉 Arquivo do Excel gerado com sucesso!")


🎉 Arquivo do Excel gerado com sucesso!
